# 01 — Node.js Fundamentals

This notebook covers the foundational concepts every Node.js developer must know for interviews.

---

## Table of Contents
1. What is Node.js?
2. The V8 Engine
3. Single-Threaded Model
4. The Event Loop (Deep Dive)
5. `process` Object
6. `global` vs `globalThis`
7. Node.js vs Browser JS
8. Interview Questions

---
## 1. What is Node.js?

**Node.js** is a **JavaScript runtime** built on Chrome's **V8 engine**. It allows you to run JavaScript on the server side.

### Key characteristics:
- **Single-threaded** with event-driven, non-blocking I/O
- Uses **libuv** under the hood for async operations (file system, networking, etc.)
- NOT a framework, NOT a language — it's a **runtime environment**

### Interview Tip:
> "Node.js is a runtime, not a framework. It uses the V8 engine to execute JavaScript and libuv to handle asynchronous I/O operations in a non-blocking, event-driven manner."

### Architecture Diagram (mental model):
```
┌─────────────────────────────────────┐
│          Your JavaScript Code       │
├─────────────────────────────────────┤
│          Node.js Bindings           │
│        (C++ bridge layer)           │
├──────────────┬──────────────────────┤
│   V8 Engine  │       libuv          │
│  (JS → Machine│  (Async I/O,        │
│    Code)     │   Event Loop,        │
│              │   Thread Pool)       │
├──────────────┴──────────────────────┤
│        Operating System             │
└─────────────────────────────────────┘
```

---
## 2. The V8 Engine

V8 is Google's open-source JavaScript engine, written in C++. It compiles JavaScript directly to **native machine code** (not bytecode interpreted at runtime like older engines).

### How V8 works:
1. **Parsing** — JS source → AST (Abstract Syntax Tree)
2. **Ignition** (Interpreter) — AST → Bytecode (quick startup)
3. **TurboFan** (Optimizing Compiler) — Hot functions → Optimized machine code
4. **Deoptimization** — If assumptions fail, falls back to bytecode

### Why this matters in interviews:
- Explains why Node.js is **fast** for I/O-bound tasks
- Explains why it's **not ideal** for CPU-heavy tasks (single thread gets blocked)
- Understanding V8's JIT compilation shows depth of knowledge

In [ ]:
// You can check your V8 version
console.log('V8 version:', process.versions.v8);
console.log('Node.js version:', process.version);
console.log('All versions:', process.versions);

---
## 3. Single-Threaded Model

Node.js runs your JavaScript on a **single thread** (the main thread). But this doesn't mean it can only do one thing at a time.

### The mental model:
- **Your JS code** → runs on 1 thread
- **I/O operations** (file read, network request, DB query) → delegated to libuv's thread pool or OS async mechanisms
- **Callbacks** → queued back on the main thread when I/O completes

### Common misconception (and interview trap):
> "Node.js is single-threaded" does NOT mean it can only handle one request at a time. The event loop allows it to handle thousands of **concurrent** connections by delegating I/O and processing callbacks when results are ready.

### When single-threaded becomes a problem:
- CPU-intensive tasks (image processing, cryptography, complex calculations)
- Solution: **Worker Threads**, **Child Processes**, or offload to a queue

In [ ]:
// Demonstrating single-threaded blocking

// This function blocks the main thread
function blockingOperation() {
    const start = Date.now();
    while (Date.now() - start < 2000) {} // Block for 2 seconds
    return 'Done blocking';
}

console.log('Before blocking:', new Date().toISOString());
console.log(blockingOperation()); // Nothing else can run during this!
console.log('After blocking:', new Date().toISOString());

In [ ]:
// Non-blocking alternative
function nonBlockingOperation(callback) {
    setTimeout(() => {
        callback('Done (non-blocking)');
    }, 2000);
}

console.log('Before non-blocking:', new Date().toISOString());
nonBlockingOperation((result) => {
    console.log(result, new Date().toISOString());
});
console.log('After non-blocking (runs immediately!):', new Date().toISOString());

---
## 4. The Event Loop — Deep Dive

This is the **#1 most asked topic** in Node.js interviews.

### What is the Event Loop?
The event loop is the mechanism that allows Node.js to perform non-blocking I/O operations despite JavaScript being single-threaded. It offloads operations to the OS kernel whenever possible.

### Phases of the Event Loop:
```
   ┌───────────────────────────┐
┌─>│         timers            │  ← setTimeout, setInterval callbacks
│  └──────────┬────────────────┘
│  ┌──────────┴────────────────┐
│  │     pending callbacks     │  ← I/O callbacks deferred to next loop
│  └──────────┬────────────────┘
│  ┌──────────┴────────────────┐
│  │       idle, prepare       │  ← internal use only
│  └──────────┬────────────────┘
│  ┌──────────┴────────────────┐
│  │         poll              │  ← retrieve new I/O events
│  └──────────┬────────────────┘
│  ┌──────────┴────────────────┐
│  │         check             │  ← setImmediate callbacks
│  └──────────┬────────────────┘
│  ┌──────────┴────────────────┐
└──┤    close callbacks        │  ← socket.on('close', ...)
   └───────────────────────────┘
```

### Microtasks vs Macrotasks:
- **Microtasks** (higher priority): `process.nextTick()`, `Promise.then/catch/finally`
- **Macrotasks**: `setTimeout`, `setInterval`, `setImmediate`, I/O callbacks

**Between every phase** of the event loop, Node.js checks and drains the microtask queue first.

In [ ]:
// CLASSIC INTERVIEW QUESTION: What is the output order?

console.log('1 - Start');

setTimeout(() => {
    console.log('2 - setTimeout');
}, 0);

Promise.resolve().then(() => {
    console.log('3 - Promise');
});

process.nextTick(() => {
    console.log('4 - nextTick');
});

console.log('5 - End');

// Answer: 1-Start, 5-End, 4-nextTick, 3-Promise, 2-setTimeout
// Why: synchronous first, then microtasks (nextTick before Promise), then macrotasks

In [ ]:
// ADVANCED: setTimeout vs setImmediate
// Inside an I/O callback, setImmediate always fires before setTimeout

const fs = require('fs');

fs.readFile(__filename, () => {
    setTimeout(() => console.log('timeout inside I/O'), 0);
    setImmediate(() => console.log('immediate inside I/O'));
});

// Output (deterministic): immediate inside I/O, then timeout inside I/O
// Because: after poll phase, check phase (setImmediate) runs before timers phase

In [ ]:
// TRICKY: process.nextTick() can starve the event loop!

// DON'T do this in production — it blocks I/O forever:
// function recursive() {
//     process.nextTick(recursive); // never lets the event loop proceed
// }

// SAFE alternative using setImmediate:
let count = 0;
function safeRecursive() {
    if (count < 5) {
        count++;
        console.log('Iteration', count);
        setImmediate(safeRecursive); // allows I/O between iterations
    }
}
safeRecursive();

### Event Loop Priority Order (memorize this!):

1. **Synchronous code** (runs first, always)
2. **`process.nextTick()`** (microtask — highest priority callback)
3. **`Promise.then()`** (microtask — after nextTick)
4. **`setTimeout(fn, 0)` / `setInterval()`** (macrotask — timers phase)
5. **`setImmediate()`** (macrotask — check phase)
6. **I/O callbacks** (macrotask — poll phase)

> **Tip:** `process.nextTick` fires before Promise microtasks because Node.js has its own microtask queue separate from the V8 Promise microtask queue, and it drains `nextTick` first.

---
## 5. The `process` Object

The `process` object is a **global** that provides information about and control over the current Node.js process. It's an instance of `EventEmitter`.

In [ ]:
// Key process properties
console.log('PID:', process.pid);
console.log('Platform:', process.platform);
console.log('Architecture:', process.arch);
console.log('Node version:', process.version);
console.log('Current directory:', process.cwd());
console.log('Memory usage:', process.memoryUsage());
console.log('Uptime (seconds):', process.uptime());

In [ ]:
// process.env — environment variables (critical for config)
console.log('PATH:', process.env.PATH?.substring(0, 80) + '...');

// Common pattern in production:
const PORT = process.env.PORT || 3000;
const NODE_ENV = process.env.NODE_ENV || 'development';
console.log(`Server would run on port ${PORT} in ${NODE_ENV} mode`);

In [ ]:
// process.argv — command-line arguments
// When you run: node app.js --port 3000 --env production
// process.argv = ['path/to/node', 'path/to/app.js', '--port', '3000', '--env', 'production']

console.log('Arguments:', process.argv);

In [ ]:
// process events — important for graceful shutdown

// Handle uncaught exceptions (last resort!)
process.on('uncaughtException', (err) => {
    console.error('Uncaught Exception:', err.message);
    process.exit(1); // Always exit after uncaughtException
});

// Handle unhandled promise rejections
process.on('unhandledRejection', (reason, promise) => {
    console.error('Unhandled Rejection at:', promise, 'reason:', reason);
});

// Graceful shutdown
process.on('SIGTERM', () => {
    console.log('Received SIGTERM. Graceful shutdown...');
    // Close DB connections, finish requests, etc.
    process.exit(0);
});

console.log('Process event handlers registered');

---
## 6. `global` vs `globalThis`

| Feature | Browser | Node.js |
|---------|---------|--------|
| Global object | `window` | `global` |
| Universal | `globalThis` | `globalThis` |
| `this` in module scope | `window` | `{}` (module.exports) |

### Interview Tip:
> In Node.js, each file is wrapped in a function (the module wrapper), so `this` at the top level is NOT `global` — it's `module.exports`. Use `globalThis` for cross-platform code.

In [ ]:
// The module wrapper — Node.js wraps every file in this:
// (function(exports, require, module, __filename, __dirname) {
//     // Your code lives here
// });

console.log('__filename:', __filename);
console.log('__dirname:', __dirname);
console.log('this === global:', this === global);       // false in a module!
console.log('this === module.exports:', this === module.exports); // true!

---
## 7. Node.js vs Browser JavaScript

| Feature | Node.js | Browser |
|---------|---------|--------|
| Runtime | V8 + libuv | V8/SpiderMonkey/etc. |
| Global | `global` / `globalThis` | `window` / `globalThis` |
| Modules | CommonJS + ESM | ESM (native) |
| DOM Access | No | Yes |
| File System | Yes (`fs`) | No (sandboxed) |
| Network (low-level) | Yes (`net`, `http`) | Fetch/XHR only |
| `this` at top level | `module.exports` | `window` |
| `require()` | Built-in | Not available |
| `process` | Available | Not available |
| `document` / `window` | Not available | Available |

---
## 8. Interview Questions & Answers

### Q1: What is Node.js and why would you use it?
**A:** Node.js is a JavaScript runtime built on the V8 engine. It's ideal for I/O-heavy applications (APIs, real-time apps, microservices) because of its non-blocking, event-driven architecture. It's not ideal for CPU-intensive tasks without Worker Threads.

### Q2: Explain the Event Loop in Node.js.
**A:** The event loop is what allows Node.js to handle asynchronous operations. It has phases (timers, pending callbacks, poll, check, close callbacks). Between phases, it drains microtask queues (nextTick, then Promises). This lets a single thread handle thousands of concurrent I/O operations.

### Q3: What's the difference between `process.nextTick()` and `setImmediate()`?
**A:** `process.nextTick()` fires before any I/O event, at the end of the current operation (microtask). `setImmediate()` fires in the "check" phase of the next event loop iteration (macrotask). Use `setImmediate()` when you want to allow I/O to happen first.

### Q4: Is Node.js single-threaded? 
**A:** JavaScript execution is single-threaded, but Node.js uses libuv's thread pool (default 4 threads) for blocking operations like file I/O, DNS lookups, and crypto. Network I/O uses OS-level async APIs (epoll, kqueue, IOCP). You can also use Worker Threads for CPU-intensive work.

### Q5: What is the `process` object?
**A:** It's a global object (EventEmitter instance) providing info and control over the Node.js process — environment variables, command-line args, memory usage, exit handling, and signal handling for graceful shutdowns.

### Q6: Why is `this` different in Node.js modules vs the browser?
**A:** Node.js wraps each file in a module wrapper function `(function(exports, require, module, __filename, __dirname) { ... })`, so `this` at the top level refers to `module.exports`, not `global`. In browsers, top-level `this` is `window`.